# vLLM client: token content, trace and metrics

Sends one request and three recorded runs to a vLLM OpenAI-compatible server (a small concurrent batch, requests sharing a long prefix, and enough load to force preemption), shows the token content of each request, and writes the two files the viewer (`app/`) replays per run:

- `runs/<name>.trace.json` (`vllm-traces/1`): per-request timings (submit, first token, every token, end) and token content (prompt + output ids and text)
- `runs/<name>.metrics.json` (`vllm-metrics/1`): `/metrics` scraped every 100 ms while the run is in flight

Config comes from `.env` (see `.env.example`), schemas are in `docs/schemas.md`.

In [ ]:
from pathlib import Path

from vllm_internals import (
    Clock, Config, Recorder, VLLMClient, build_trace, run_batch, save_json, show_tokens, summary,
)

cfg = Config.from_env()
client = VLLMClient(cfg)
model = await client.resolve_model()
OUT = Path("../runs")  # the repo root's runs/ (Jupyter starts the kernel in notebooks/)
print(f"{cfg.base_url} · {model}")

## Single request

Prompt ids come from `/tokenize` (chat template applied), output ids and text from the streamed `logprobs` (`return_tokens_as_token_ids`). Hover a token for its id, position and logprob.

In [ ]:
r = await client.chat(
    [{"role": "user", "content": "Explain paged attention in two sentences."}], max_tokens=96
)
if not r.ok:
    raise RuntimeError(r.error)
vocab = await client.attach_prompt_tokens([r])
print(f"ttft {r.t_first - r.t_submit:.3f}s · {len(r.token_t)} tokens in {r.t_end - r.t_submit:.2f}s · {r.finish_reason}")
show_tokens(r, vocab)

## Concurrent batch

Every request shares one system prompt, so with prefix caching (on by default in vLLM V1) the later ones hit the cache on its whole blocks. The server reports `cached_tokens` per request only when started with `--enable-prompt-tokens-details`. A concurrency above the server's `--max-num-seqs` shows up as a waiting queue.

In [ ]:
SYSTEM = {
    "role": "system",
    "content": "You are a terse systems engineer. Answer in at most three sentences, plain text, no lists. " * 3,
}
QUESTIONS = [
    "What is a KV cache block?",
    "Why does prefill cost more than a decode step?",
    "What does continuous batching change?",
    "When does vLLM preempt a request?",
    "What is prefix caching?",
    "What limits the batch size?",
    "Why is TTFT higher under load?",
    "What is chunked prefill?",
]
conversations = [[SYSTEM, {"role": "user", "content": q}] for q in QUESTIONS]

with Recorder(cfg.metrics_url, interval_s=0.1, headers=cfg.headers) as rec:
    clock = Clock()
    reqs = await run_batch(client, conversations, concurrency=8, max_tokens=128, clock=clock)
vocab = await client.attach_prompt_tokens(reqs)

for q in reqs:
    if q.error:
        print(f"#{q.idx} failed: {q.error}")
print(rec.describe())
show_tokens(reqs[-1], vocab)

In [ ]:
def save_run(name, reqs, rec, clock, vocab, **run):
    trace = build_trace(reqs, clock=clock, model=model, vocab=vocab, run={"label": name, **run})
    paths = [save_json(trace, OUT / f"{name}.trace.json"), rec.save(OUT / f"{name}.metrics.json")]
    s = summary(trace)
    print(f"ttft p50/p95 {s['ttft_p50']:.3f}/{s['ttft_p95']:.3f}s · "
          f"itl p50/p95 {s['itl_p50'] * 1e3:.1f}/{s['itl_p95'] * 1e3:.1f}ms · "
          f"{s['out_tok_per_s']:.0f} out tok/s · {s['errors']} errors")
    for p in paths:
        print(p.resolve())


save_run("batch", reqs, rec, clock, vocab, concurrency=8, max_tokens=128)
print("start the viewer (cd app && npm run dev) and drop both files on it")

## The KV pool

The two runs below are sized from the server's pool, read from `cache_config_info` on `/metrics`: `block_size` tokens per block, `num_gpu_blocks` blocks per engine core. With data parallel every core has its own pool and its own prefix cache.

In [ ]:
import math

scrape = Recorder(cfg.metrics_url, headers=cfg.headers).scrape()
cache = scrape["info"]["cache_config_info"]
BLOCK, N_BLOCKS = int(cache["block_size"]), int(cache["num_gpu_blocks"])
ENGINES = max(1, len(scrape["e"]))
MAX_LEN = next(m["max_model_len"] for m in (await client.http.get("/v1/models")).json()["data"] if m["id"] == model)
print(f"{ENGINES} engine core(s) · {N_BLOCKS:,} blocks of {BLOCK} tokens each "
      f"({N_BLOCKS * BLOCK:,} token slots per core) · max_model_len {MAX_LEN:,}")

## Long shared prefix

Every request carries the same long document in front of its question. Prefix caching only reuses whole blocks, chained from the first one, so the document is sized to several `block_size` blocks. One request per engine core goes first (blocks are cached once computed, so requests prefilled in the same step all miss), then the rest hit the document's blocks: in the viewer they show up as shared (ref > 1) blocks and as prefix-cache-hit slots, and stay cached and free once their readers finish.

Writes `runs/prefix.trace.json` and `runs/prefix.metrics.json`.

In [ ]:
NOTES = (
    "The scheduler admits a waiting request only while the pool has free blocks for its prompt. "
    "Every running sequence decodes one token per engine step, and takes a new block each time its last one fills. "
    "Prefix caching hashes every full block of a prompt chained to the hash of the block before it, "
    "so a block is reused only together with everything in front of it. "
    "A cached block whose last reader finished stays in the free queue, still hashed, until a fresh allocation evicts it. "
    "Chunked prefill spreads a long prompt over several steps, so the running sequences keep decoding meanwhile. "
)
per_note = len(await client.tokenize([{"role": "user", "content": NOTES}]))
# several whole blocks, with room left in the context and in the pool for the rest of each request
target = min(max(4 * BLOCK, 2048), MAX_LEN // 2, N_BLOCKS * BLOCK // 4)
DOC = "\n\n".join(f"Note {i + 1}. {NOTES}" for i in range(math.ceil(target / per_note)))
LONG_QUESTIONS = [
    "When does the scheduler admit a waiting request?",
    "How often does a running sequence take a new block?",
    "Why is a block reused only with everything in front of it?",
    "What happens to a cached block once its last reader finishes?",
    "What does chunked prefill change for running sequences?",
    "Which notes talk about the free queue?",
    "Summarise note 3 in one sentence.",
    "What evicts a cached block?",
    "How many notes are there?",
    "What is hashed by prefix caching?",
    "Does a running sequence decode more than one token per step?",
    "What keeps decoding while a long prompt is prefilled?",
]
long_convs = [
    [{"role": "system", "content": f"Answer from these notes, in at most three sentences.\n\n{DOC}"},
     {"role": "user", "content": q}]
    for q in LONG_QUESTIONS
]

with Recorder(cfg.metrics_url, interval_s=0.1, headers=cfg.headers) as rec:
    clock = Clock()
    reqs = await run_batch(client, long_convs[:ENGINES], concurrency=ENGINES, max_tokens=128, clock=clock)
    reqs += await run_batch(client, long_convs[ENGINES:], concurrency=12, max_tokens=128, clock=clock,
                            first_idx=ENGINES)
vocab = await client.attach_prompt_tokens(reqs)

for q in reqs:
    if q.error:
        print(f"#{q.idx} failed: {q.error}")
print(f"{len(reqs[0].prompt_ids):,} prompt tokens per request, "
      f"{len(reqs[0].prompt_ids) // BLOCK} whole blocks of them")
print(rec.describe())
save_run("prefix", reqs, rec, clock, vocab, concurrency=12, max_tokens=128)

## Preemption

vLLM admits a request once its prompt fits the pool and takes decode blocks as it goes, so sequences that together outgrow the pool run it dry mid-decode. The scheduler then preempts the newest running sequence: its blocks are freed, it goes back to the waiting queue and is recomputed later. 64 sequences (keep it at or below `max_num_seqs` × engine cores) each generate their share of 1.5× the pool, `ignore_eos` so none stops early. In the viewer: red preemption ticks on the engine timeline, KV utilisation pinned near 100%, and a gap in the preempted requests' token arrivals.

A production-size pool is too big to outgrow within `max_model_len`; the cell then prints the `--num-gpu-blocks-override` to restart vLLM with (rerun from "The KV pool" afterwards). Writes `runs/preempt.trace.json` and `runs/preempt.metrics.json`.

In [ ]:
PREEMPT_SEQS = 64
pool = N_BLOCKS * BLOCK * ENGINES
gen = math.ceil(1.5 * pool / PREEMPT_SEQS)

if gen > MAX_LEN - 128:  # 128: room for the prompt
    # a pool that ~1k generated tokens per sequence outgrow by half, and still holds one max_model_len request
    blocks = max(math.ceil(MAX_LEN / BLOCK), PREEMPT_SEQS * 1024 * 2 // (3 * BLOCK * ENGINES))
    print(f"{PREEMPT_SEQS} sequences cannot outgrow {pool:,} token slots within max_model_len {MAX_LEN:,}.\n"
          f"Restart vLLM with --num-gpu-blocks-override {blocks} and rerun from \"The KV pool\".")
else:
    preempt_convs = [
        [{"role": "user", "content": f"Tell a long story about request number {i}."}]
        for i in range(PREEMPT_SEQS)
    ]
    with Recorder(cfg.metrics_url, interval_s=0.1, headers=cfg.headers) as rec:
        clock = Clock()
        reqs = await run_batch(client, preempt_convs, concurrency=PREEMPT_SEQS, max_tokens=gen,
                               clock=clock, extra_body={"ignore_eos": True})
    vocab = await client.attach_prompt_tokens(reqs)

    for q in reqs:
        if q.error:
            print(f"#{q.idx} failed: {q.error}")
    print(f"{PREEMPT_SEQS} × {gen:,} generated tokens against {pool:,} token slots")
    print(rec.describe())
    save_run("preempt", reqs, rec, clock, vocab, concurrency=PREEMPT_SEQS, max_tokens=gen)

In [ ]:
await client.aclose()